In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import QuantileTransformer
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score

import warnings
warnings.filterwarnings("ignore")

In [ ]:
#prepare + load df
data = pd.read_csv("bl_progression_data.csv")
emci_lmci = data[data["BASELINE_DX"].isin(["EMCI", "LMCI"])].copy()

exclude_cols = ['RID','VISCODE','BASELINE_DX','FINAL_DX',
                'PROGRESSION_MONTH','PROGRESSED','TOTAL_VISITS','DX','DX_bl']
metabolite_cols = [c for c in emci_lmci.columns if c not in exclude_cols]

X_all = emci_lmci[metabolite_cols].to_numpy()
y_all = (emci_lmci["BASELINE_DX"] == "LMCI").astype(int).to_numpy()

print(f"Total patients: {len(X_all)}, Total features: {len(metabolite_cols)}")
print(f"EMCI: {(y_all==0).sum()}, LMCI: {(y_all==1).sum()}")

Total patients: 643, Total features: 104
EMCI: 222, LMCI: 421


In [ ]:
#data split
# 80/20 split (outer split)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_all, y_all,
    test_size=0.2,
    stratify=y_all,
    random_state=42  # fixed
)

In [ ]:
#100-fold cv
skf = StratifiedKFold(n_splits=100, shuffle=True, random_state=42)

#mutual info random state
def custom_MI(X, y):
    return mutual_info_classif(X, y, random_state=42)

In [ ]:
#inner cv loop
def run_fold(X_train, X_val, y_train, y_val, metabolite_cols):

    #scaling on training folds
    scaler = QuantileTransformer(output_distribution="normal", random_state=42)
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    #feature selection 1 (MI)
    selector1 = SelectKBest(score_func = custom_MI, k=30)
    X_train_kbest = selector1.fit_transform(X_train, y_train)
    selected_kbest = selector1.get_support(indices=True)

    #feature selection 2 (RFE)
    log_reg = LogisticRegression(penalty="l1", solver="liblinear", random_state=42)
    rfe = RFE(log_reg, n_features_to_select=15)
    X_train_rfe = rfe.fit_transform(X_train_kbest, y_train)

    #collect feature names
    selected_features = [
        metabolite_cols[selected_kbest[i]]
        for i in range(len(selected_kbest))
        if rfe.support_[i]
    ]

    #train classif model on final 15
    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42
    )
    rf.fit(X_train_rfe, y_train)

    #validate
    X_val_kbest = selector1.transform(X_val)
    X_val_rfe = rfe.transform(X_val_kbest)

    y_pred = rf.predict(X_val_rfe)

    metrics = [
        accuracy_score(y_val, y_pred),
        f1_score(y_val, y_pred,average='binary'),
        recall_score(y_val, y_pred,average='binary')
    ]

    return selected_features, metrics

In [ ]:
#data storage
all_selected_features = []
cv_metrics = []

for train_idx, val_idx in skf.split(X_train_full, y_train_full):

    X_train = X_train_full[train_idx]
    X_val = X_train_full[val_idx]
    y_train = y_train_full[train_idx]
    y_val = y_train_full[val_idx]

    features, metrics = run_fold(
        X_train, X_val, y_train, y_val, metabolite_cols
    )

    all_selected_features.extend(features)
    cv_metrics.append(metrics)

In [ ]:
cv_metrics = np.array(cv_metrics)
print("CV accuracy: ", cv_metrics[:,0].mean().round(3), "±", cv_metrics[:,0].std().round(3))
print("CV F1:       ", cv_metrics[:,1].mean().round(3), "±", cv_metrics[:,1].std().round(3))
print("CV recall:   ", cv_metrics[:,2].mean().round(3), "±", cv_metrics[:,2].std().round(3))


CV accuracy:  0.819 ± 0.165
CV F1:        0.846 ± 0.151
CV recall:    0.822 ± 0.199


In [ ]:
#sanity check
print(skf.get_n_splits())

100


In [ ]:
freqs = Counter(all_selected_features)
#create df for metab column and count column
df_features = pd.DataFrame(freqs.items(), columns=["Metabolite", "Count"])
#create a frequency column based off count
df_features["Frequency (%)"] = df_features["Count"] / (100) * 100  # 100 skf folds , hardcoded
df_features = df_features.sort_values(['Frequency (%)', 'Count'], ascending=False).reset_index(drop=True)

# Select top 15 most frequent
top15 = df_features.head(15)['Metabolite'].tolist()

print(df_features.head(15).to_string(index=False))

               Metabolite  Count  Frequency (%)
            GLYCERIC_ACID    100          100.0
   X2_HYDROXYBUTYRIC_ACID    100          100.0
X__2_METHYLPENTANOIC_ACID    100          100.0
                     C8_0    100          100.0
                  X3_DHCA    100          100.0
                  ALLOLCA    100          100.0
                    GHDCA    100          100.0
                      HCA    100          100.0
                    TAMCA    100          100.0
                    C12_0     92           92.0
                    C14_0     81           81.0
                     C4_0     81           81.0
          L_ASPARTIC_ACID     74           74.0
         C18_3_CIS_6_9_12     71           71.0
                 L_LYSINE     60           60.0


In [ ]:
#final on development
scaler_final = QuantileTransformer(output_distribution="normal", random_state=42)
X_train_scaled = scaler_final.fit_transform(X_train_full)
X_test_scaled = scaler_final.transform(X_test)

#convert metabolite names → indices
feature_indices = [metabolite_cols.index(f) for f in top15]

#select only final 15 features
X_train_final = X_train_scaled[:, feature_indices]
X_test_final = X_test_scaled[:, feature_indices]

#train final model
rf_final = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42
)

rf_final.fit(X_train_final, y_train_full)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 T

In [ ]:
y_pred_test = rf_final.predict(X_test_final)

test_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_test),
    "F1": f1_score(y_test, y_pred_test, average='binary'),
    "Recall": recall_score(y_test, y_pred_test, average='binary')
}

print("Final Test Metrics:")
print(test_metrics)


Final Test Metrics:
{'Accuracy': 0.7829457364341085, 'F1': 0.8227848101265823, 'Recall': 0.7738095238095238}


In [ ]:
# save csvs

pd.DataFrame(cv_metrics, columns=['Accuracy', 'F1', 'Recall']
).to_csv('K100_cv_metrics.csv', index=False)

pd.DataFrame([{
    'Accuracy':    accuracy_score(y_test, y_pred_test),
    'F1':   f1_score(y_test, y_pred_test, average='binary'),
    'Recall': recall_score(y_test, y_pred_test, average='binary')
}]).to_csv('K100_test_metrics.csv', index=False)

In [ ]:
name_map = {
    'GLYCERIC_ACID': 'Glyceric acid',
    'X2_HYDROXYBUTYRIC_ACID': '2-Hydroxybutyric acid',
    'X__2_METHYLPENTANOIC_ACID': '2-Methylpentanoic acid',
    'C8_0': 'Caprylic acid',
    'X3_DHCA': '3-Dehydrocholic acid',
    'ALLOLCA': 'Allolithocholic acid',
    'GHDCA': 'Glycohyodeoxycholic acid',
    'HCA': 'Muricholic/Hyocholic acid',
    'TAMCA': 'Tauro-muricholic acid',
    'C12_0': 'Dodecanoic acid',
    'C14_0': 'Myristic acid',
    'C4_0': 'Butyric acid',
    'L_ASPARTIC_ACID': 'L-Aspartic acid',
    'C18_3_CIS_6_9_12': 'Gamma-linolenic acid',
    'L_LYSINE': 'L-Lysine'
}
df_panel = df_features.head(15).copy()
df_panel['Proper Name'] = df_panel['Metabolite'].map(name_map)
df_panel.to_csv('K100_top15_panel.csv', index=False)

print(df_panel.to_string(index=False))

               Metabolite  Count  Frequency (%)               Proper Name
            GLYCERIC_ACID    100          100.0             Glyceric acid
   X2_HYDROXYBUTYRIC_ACID    100          100.0     2-Hydroxybutyric acid
X__2_METHYLPENTANOIC_ACID    100          100.0    2-Methylpentanoic acid
                     C8_0    100          100.0             Caprylic acid
                  X3_DHCA    100          100.0      3-Dehydrocholic acid
                  ALLOLCA    100          100.0      Allolithocholic acid
                    GHDCA    100          100.0  Glycohyodeoxycholic acid
                      HCA    100          100.0 Muricholic/Hyocholic acid
                    TAMCA    100          100.0     Tauro-muricholic acid
                    C12_0     92           92.0           Dodecanoic acid
                    C14_0     81           81.0             Myristic acid
                     C4_0     81           81.0              Butyric acid
          L_ASPARTIC_ACID     74      

In [ ]:
hmdb_map = {
    'GLYCERIC_ACID': 'HMDB00139',
    'X2_HYDROXYBUTYRIC_ACID': 'HMDB00008',
    'X__2_METHYLPENTANOIC_ACID': 'HMDB31580',
    'C8_0': 'HMDB00482',
    'X3_DHCA': 'HMDB00502',
    'ALLOLCA': 'HMDB00713',
    'GHDCA': None,
    'HCA': 'HMDB00760',
    'TAMCA': None,
    'C12_0': 'HMDB00638',
    'C14_0': 'HMDB00806',
    'C4_0': 'HMDB00039',
    'L_ASPARTIC_ACID': 'HMDB00191',
    'C18_3_CIS_6_9_12': 'HMDB03073',
    'L_LYSINE': 'HMDB00182'
}
